# Find the Best Model

**IDETC-CIE 2026 · EngiBench hands-on workshop**

You have a bank of generative models for the same engineering design problem.
They are labelled `Model A`, `Model B`, ... and nothing else. Your job is to work
out which one you would ship, and to be able to defend it.

You have three metrics to work with. They are the three that generative-design
papers report most often, and you will notice they are also the three that are
cheapest to compute. That is not a coincidence, and it is most of what this
session is about.

**The rules:**
1. Look at the designs before you compute anything.
2. Compute the metrics you can afford.
3. Commit to a winner, in writing, with a reason. Your commitment gets hashed.
4. *Then* we start revealing things.

A team that says *"these metrics don't separate this bank, I need more"* has
won the exercise. The commitment cell needs a pick, but
**"I would not ship any of these because ___" is an accepted answer.**

**Before you edit anything:** File → Save a copy in Drive. This notebook opens read-only from GitHub.

---
## Step 0 · Setup

Run this once, then restart the runtime when it tells you to.

In [ ]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    def pip_install(pkgs):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

    pip_install(["engibench[all]", "cryptography", "matplotlib", "pandas", "scikit-learn"])
    pip_install(["git+https://github.com/IDEALLab/EngiOpt.git@feat/idetc26-workshop#egg=engiopt"])
    print("Install complete. Runtime → Restart session, then continue from the next cell.")
else:
    print("Using the current environment.")

In [ ]:
import pandas as pd
import torch as th

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# A GPU speeds up sampling from the trained models in the bank. It does *not*
# speed up the physics: this problem's optimizer is CPU-bound topology
# optimization, so the expensive metrics cost the same either way.
print("GPU available:", th.cuda.is_available())

---
## Step 1 · Open the challenge

Put your team's name in. It fixes which model gets which letter, so your bank is
ordered differently from the team next to you — you cannot shortcut this by
comparing letters across tables, and the same name always gives you the same
bank if you rerun.

In [ ]:
from engiopt.workshops.idetc26 import Challenge

TEAM = "orange"          # <- your team name
PROBLEM = "beams2d"      # <- your problem

ch = Challenge.open(PROBLEM, team=TEAM)

### What the problem is

`beams2d` is cantilever topology optimization. You are given a volume budget and
a few other conditions, and you must lay out material in a 50×100 grid so the
beam is as stiff as possible. Every model in your bank has been asked for
designs under the **same 50 sets of conditions**, drawn from a frozen evaluation
spec — so anything you see is a difference between models, never a difference in
what they were asked.

---
## Round 1 · The eyeball test

Before any number exists, look at the designs.

Visual inspection is the metric every practitioner actually uses, every paper
includes as a figure, and no paper reports as a number. It takes four seconds
and it catches things no column in this notebook catches. It is also
unreproducible, unaggregatable, trivially cherry-picked, and you are seeing four
designs out of fifty chosen by whoever wrote this cell.

Hold both of those thoughts. Rank the bank by eye anyway.

In [ ]:
fig = ch.gallery(n_per_model=4)

**Fill in:** your ranking by eye, best first. Two minutes, no discussion with other teams.

In [ ]:
print("Your bank, in order:", ch.bank.labels)

# START FILL -- reorder this list, best first
eyeball_ranking = list(ch.bank.labels)
# END FILL

assert sorted(eyeball_ranking) == sorted(ch.bank.labels), "List every model exactly once."
print("Recorded. You will be asked about this again.")

---
## Round 2 · The metrics you can afford

Three columns:

| metric | asks | direction |
|---|---|---|
| `mmd` | Does the set of generated designs look like the set of real ones? | lower is better |
| `dpp` | Is the model producing varied designs, or repeating itself? | higher is better |
| `viol` | What fraction of designs break a constraint or miss the volume budget? | lower is better |

**No simulator runs in this cell.** That is why it takes seconds instead of
twenty minutes, and it is why these are the metrics you see in papers.

In [ ]:
board = ch.board()
board.round(4)

Which model tops each column?

In [ ]:
ch.winners(board)

**Fill in:** commit to a winner.

Your pick and your reason get hashed and written to `verdict.json`. Read the
hash out when your team presents — it is how the room knows you did not revise
after the reveal. That is the practice this whole session is arguing for, so we
may as well run it on ourselves.

In [ ]:
# START FILL
winner = "Model A"
why = "..."
metric_ranking = list(ch.bank.labels)   # reorder, best first
# END FILL

ch.submit(winner=winner, why=why, ranking=metric_ranking, eyeball_ranking=eyeball_ranking)

---
## Reveal 0 · Your own two rankings

Nothing has been unsealed. This uses only data your team produced in the last
fifteen minutes.

In [ ]:
comparison = pd.DataFrame({
    "by eye": {label: i + 1 for i, label in enumerate(eyeball_ranking)},
    "by metric": {label: i + 1 for i, label in enumerate(metric_ranking)},
}).sort_values("by metric")
comparison["moved"] = (comparison["by eye"] - comparison["by metric"]).abs()
comparison

If those two columns disagree, one of your two procedures is wrong and you do
not yet know which. Most teams find they disagree badly. **Discuss for two
minutes: which one do you actually trust, and what would settle it?**

---
## Reveal 1 · The seed lottery

Same models. Same three metrics. Same conditions. The only thing that changes is
the random seed the models sample at.

Still nothing unsealed, still no simulator.

In [ ]:
lottery = ch.seed_lottery()
lottery

Look along a row. If a model's rank moves between seeds, then **every
single-seed ranking is a coin flip you did not know you were making** — and most
published comparisons report one seed.

This is the cheapest possible criticism of your own result and almost nobody
runs it.

---
## Reveal 2 · The columns you were not given

Seven more metrics. **Every one of them is cheap.** No simulator, no cluster, no
waiting. They were affordable the entire time; you simply were not handed them.

| metric | asks | direction |
|---|---|---|
| `novelty` | How far is each design from the nearest *training* design? | higher is better |
| `cond_err` | Did the design hit the volume fraction it was asked for? | lower is better |
| `pixel_vendi` | How many *effectively distinct* designs are in the set? | higher is better |
| `pca_mmd` | `mmd`, but measured in a PCA subspace instead of raw pixels | lower is better |
| `pca_vendi` | `pixel_vendi`, likewise moved off raw pixels | higher is better |
| `gen_seconds` | What did it cost to generate the set? | lower is better |
| `params` | How big is the model? For the lookup table, that is its training set. | lower is better |

In [ ]:
withheld = ch.withheld()
withheld.round(4)

In [ ]:
full_board = pd.concat([board, withheld], axis=1)
ch.winners(full_board)

Two things to notice, and they are different kinds of problem.

**`novelty` is the memorization check.** Every distribution metric — `mmd`,
`pca_mmd`, and the precision/coverage family too — is *optimized* by copying the
training set. A lookup table beats every one of them. Only distance-to-training
separates a model that learned the manifold from one that memorized points on
it, and it is one line of code.

**`dpp` and `pixel_vendi` both claim to measure diversity, and only one of them
is readable.** Most of the `dpp` column prints as `0.0000`. Those are not ties —
run the cell below and look at the actual values.

In [ ]:
board[["dpp"]].map(lambda v: f"{v:.3e}").join(withheld[["pixel_vendi"]].round(2))

`dpp` is the determinant of a 50×50 kernel matrix — a product of fifty numbers
below one — so it lands anywhere between `1e0` and `1e-290`. It is not wrong.
It is **unreportable**: at any precision a paper's table would use, distinct
models render as identical zeros. `pixel_vendi` measures the same property as an
effective sample count (1 means "one design repeated", 50 means "fifty distinct
designs"), on a scale that survives being written down.

A metric can fail by being uninformative. It can also fail by being
uncommunicable, and that failure is invisible in the code.

---
## Reveal 2b · What does a diversity number *mean*?

You now have three diversity columns disagreeing with each other. Before
arguing about which is right, calibrate them: score models whose answer you
already know.

- `collapsed` — one design, repeated fifty times. The floor of any diversity
  column.
- `noise_doped` — real optimal designs with noise added. Strictly worse designs,
  by construction.
- `volume_only` — hits the volume budget exactly, with material arranged to
  carry no load. Perfect feasibility, useless structure.

These are **not** in your bank and they are never ranked. They are a scale bar.

In [ ]:
reference = ch.reference_row(metrics=("dpp", "pixel_vendi", "novelty", "viol"))
reference.round(4)

Read `noise_doped` against the models in your bank. Adding noise to a real
optimal design cannot make it a better design — but look at what it does to
`pixel_vendi`, and to `dpp`.

**A diversity metric that rewards corruption is not measuring diversity.** It is
measuring entropy, and entropy is free.

---
## Reveal 2c · The space you measured in

Every column so far compared designs **pixel by pixel**. That is a modelling
choice. Nobody declares it, and it is not obviously the right one: two designs
that differ by a one-pixel shift are nearly identical structurally and far apart
in pixels.

`pca_mmd` already hinted at this — same question, linear subspace instead of raw
pixels, different answer. Now go further and measure in the latent space of an
autoencoder trained on **real optimal designs for this problem**, with a
performance-prediction constraint that forces it to keep the variation that
changes how a design performs.

These columns are **cheap**: an encode and a decode, the same seconds the pixel
columns cost. What you are buying is not compute, it is a space.

| metric | asks |
|---|---|
| `lv_mmd` | `mmd`, measured on the manifold instead of on pixels |
| `lv_coverage` | What fraction of real design modes did the generator reach? |
| `lv_vendi` | How many effectively distinct designs, counted structurally? |
| `lv_paired_distance` | Did each design answer the condition it was actually asked for? |

All four **encode only**. Two further columns — `lv_residual` (how far off the
manifold each design sits) and `lv_dual_gap` (whether that deviation was
performance-relevant or cosmetic) — additionally need to *decode*, and are held
back while the instruments are retrained. That is worth saying out loud rather
than hiding: a metric that depends on a fitted instrument inherits that
instrument's version history, which is a cost the pixel columns do not have.

In [ ]:
manifold = ch.manifold()
manifold.round(4)

The instrument is pinned in the frozen spec, not chosen here. Which autoencoder
you measure in decides every number above, so a latent column is only comparable
between two rows encoded by the *same* instrument — which is why it has to be
declared, and why the spec carries its fingerprint and revision.

In [ ]:
ch.instrument()

In [ ]:
full_board = pd.concat([full_board, manifold], axis=1)
ch.winners(full_board)

**Rerun the reference row in this space** and compare it to the pixel one you
just computed. The noise-doped instrument is the test: pixel diversity rises
when you corrupt a real design, and the question is whether the latent version
does the same thing.

In [ ]:
ch.reference_row(metrics=("pixel_vendi", "pca_vendi", "lv_vendi")).round(4)

---
## Optional · Run the simulator yourself

Everything so far cost seconds. The next section reveals a board somebody else
computed, and the reason it was precomputed is that **it costs hours**.

You do not have to take that on faith. Run it on two or three designs and watch
the clock. Each sample runs one optimization and two simulations.

The first call only prints an estimate — it does not run anything. Pass
`confirm=True` when you have decided to wait.

In [ ]:
# What would it cost? (This does not run the simulator.)
ch.run_physics(n_samples=3)

In [ ]:
# START FILL -- pick how much you are willing to wait for
my_models = ch.bank.labels[:2]   # which models
my_samples = 2                   # conditions per model
# END FILL

mine = ch.run_physics(labels=my_models, n_samples=my_samples, confirm=True)
mine.round(3)

Two things to take from that number.

**Your estimate is not the sealed board.** That one is computed at the spec's
full 50 conditions; yours is 2 or 3. When the board is unsealed, compare the two
for a model you ran — the gap between a cheap estimate of an expensive metric
and the expensive metric is exactly the thing nobody reports.

**Now multiply.** A hyperparameter sweep is fifty configurations. Five seeds
each. Three problems. At the rate you just watched, that is the reason every
paper you have read reports `mmd` and not `cog`.

---
## Reveal 3 · The physics

Now the simulator. `iog`, `cog` and `fog` measure how far each design is from an
optimal one, before and after re-optimization — the thing you actually care
about, and the thing nobody can afford to put in a rebuttal.

This board was computed ahead of time and sealed into the repository with its
hash published beside it, so you can check afterwards that the answer was fixed
before yours was.

Your facilitator will read out the passphrase.

In [ ]:
# START FILL
passphrase = "..."
# END FILL

physics = ch.unseal_physics(passphrase)
physics.round(3)

In [ ]:
final = pd.concat([full_board, physics], axis=1)
ch.winners(final)

---
## Reveal 4 · Who they actually were

In [ ]:
ch.identities()

There was no trick in the bank. Every entry is a model somebody might really
ship: trained checkpoints across five architectures, plus two dataset-fitted
baselines that are **not** jokes —

- a **lookup table** (`knn_retrieval`) that returns the nearest training design,
  rescaled to hit the requested volume budget
- a **ridge regression** (`linear_regression`) from conditions straight to
  pixels, with no generative model in it at all

Neither needed a GPU. The lookup table's checkpoint *is* its training set, and
it costs about a thousandth of what the diffusion model costs to sample.

Look at where they landed. If a retrieval baseline tops your board, that is a
result and not a broken leaderboard — it is exactly what Habibi et al. found on
this task. **The honest version of the lesson is harder than a planted fraud:
nothing here was rigged, and the metrics still could not tell you what to
ship.**

Two of the pairs in the bank differ only by training seed. Find them in the
table. Any gap you attributed to architecture that is smaller than that pair's
gap was never about architecture.

---
## Debrief · Write the recipe you would actually trust

You now have nineteen columns and a demonstration that any one of them can be
gamed. So write down what you would require before believing a claim that model
X is better than model Y.

In [ ]:
# START FILL
recipe = {
    "metrics": ["..."],              # which columns, and why each earns its place
    "n_seeds": 1,                    # how many seeds before you believe a ranking
    "aggregate": "median",           # median or mean, and what you report alongside it
    "gates": ["..."],                # what disqualifies a model outright, regardless of score
    "visual_protocol": "...",        # how many designs, chosen how, shown to whom
}
# END FILL

import json
print(json.dumps(recipe, indent=2))

Trade recipes with the team next to you and try to break theirs. Specifically:

1. Which of the models in this bank would still pass their gates?
2. What would it cost to run their recipe on a new problem?
3. Their `visual_protocol` is the interesting one. "We looked at some designs" is
   exactly the unreproducible part, and it is what every paper does. Can they
   write one you could actually replicate?

---

**Take it further:** `python -m engiopt.evaluate --list-metrics` shows every
column the benchmark can compute. `BRING_YOUR_OWN_PROBLEM.md` walks through
running this arc on a problem of your own.